In [1]:
import json
from pathlib import Path
from pprint import pprint
from uuid import uuid4
from datetime import datetime

from src.context.contextbuilder import ContextBuilder
from src.engine.LlmProviderManager import LlmProvider
from src.models.MemoryEvent import MemoryEvent


def load_config() -> dict:
    config_path = "config.json"

    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)


def make_event(
    content: str,
    event_type: str,
    source: str,
    step: int,
) -> MemoryEvent:

    return MemoryEvent(
        id=uuid4(),
        event_type=event_type,
        content=content,
        source=source,
        step=step,
        timestamp=datetime.now(),
        metadata={},
    )


def print_message(index: int, message: dict) -> None:

    print()
    print(f"{'=' * 20} MESSAGE {index} {'=' * 20}")

    print("ROLE:")
    print(message.get("role"))

    print("\nCONTENT:")
    print(message.get("content"))

    if "tool_calls" in message:
        print("\nTOOL CALLS:")
        pprint(message["tool_calls"])


def main():

    print("=" * 80)
    print("EVANA CONTEXT BUILDER REAL TEST")
    print("=" * 80)

    # ---------------------------------------------------------
    # Config
    # ---------------------------------------------------------

    config = load_config()

    print("\n[CONFIG]")
    print(f"Provider: " f"{config['llm']['provider']}")

    print(f"Model: " f"{config['llm']['provider_config']['model_name']}")

    print(
        f"Context length: "
        f"{config['llm']['provider_config']['generation_config']['num_ctx']}"
    )

    # ---------------------------------------------------------
    # REAL LLM PROVIDER
    # ---------------------------------------------------------

    print("\n[LLM PROVIDER]")

    llm = LlmProvider(config)

    print(f"Loaded provider: " f"{llm.provider_name}")

    print(f"Model object: " f"{type(llm.model).__name__}")

    # ---------------------------------------------------------
    # CONTEXT BUILDER
    # ---------------------------------------------------------

    builder = ContextBuilder(
        config=config,
        llm_provider=llm,
    )

    print("\n[CONTEXT BUILDER]")
    print(f"Token budget: " f"{builder.tokenbudget.budget}")

    print(f"Safe margin: " f"{builder.tokenbudget.safe_margin}")

    print(f"Initial chars/token: " f"{builder.tokenbudget.chars_per_token}")

    # ---------------------------------------------------------
    # EVENTS
    # ---------------------------------------------------------

    events = [
        make_event(
            content="Build a Flask shop website.",
            event_type="user_input",
            source="user",
            step=1,
        ),
        make_event(
            content="I will inspect the project structure first.",
            event_type="assistant_message",
            source="assistant",
            step=2,
        ),
        make_event(
            content=('{"tool_name":"Read",' '"arguments":{"path":"shop/app.py"}}'),
            event_type="tool_call",
            source="assistant",
            step=3,
        ),
        make_event(
            content=(
                "from flask import Flask\n"
                "\n"
                "app = Flask(__name__)\n"
                "\n"
                "@app.route('/')\n"
                "def home():\n"
                "    return 'Shop'"
            ),
            event_type="tool_result",
            source="tool",
            step=4,
        ),
        make_event(
            content="I inspected shop/app.py successfully.",
            event_type="assistant_message",
            source="assistant",
            step=5,
        ),
    ]

    # ---------------------------------------------------------
    # EXTERNAL AGENT STATE
    # ---------------------------------------------------------

    task = {
        "goal": "Complete the Flask shop website",
    }

    active_skills = {
        "filesystem": {
            "enabled": True,
        },
        "python": {
            "enabled": True,
        },
    }

    agent_state = {
        "status": "working",
        "iteration": 5,
    }

    progress = {
        "completed": [
            "Inspected project structure",
            "Inspected shop/app.py",
        ],
        "pending": [
            "Modify shop/app.py",
            "Add product images",
            "Run tests",
        ],
        "failed": [],
    }

    # ---------------------------------------------------------
    # BUILD
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print("BUILDING CONTEXT...")
    print("=" * 80)

    messages = builder.build_context(
        events=events,
        task=task,
        active_skills=active_skills,
        agent_state=agent_state,
        progress=progress,
    )

    # ---------------------------------------------------------
    # RAW WINDOW STATE
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print("CONTEXT WINDOW STATE")
    print("=" * 80)

    print("\nSYSTEM:")
    pprint(builder.window.system)

    print("\nTASK:")
    pprint(builder.window.task)

    print("\nACTIVE SKILLS:")
    pprint(builder.window.active_skills)

    print("\nAGENT STATE:")
    pprint(builder.window.agent_state)

    print("\nPROGRESS:")
    pprint(builder.window.progress)

    print("\nLAST ACTION:")
    pprint(builder.window.last_action)

    print("\nLAST OBSERVATION:")
    pprint(builder.window.last_observation)

    print("\nCONVERSATION:")
    pprint(builder.window.conversation)

    # ---------------------------------------------------------
    # RAW get_prompt()
    # ---------------------------------------------------------

    raw_messages = builder.window.get_prompt()

    print("\n" + "=" * 80)
    print("RAW ContextWindow.get_prompt()")
    print("=" * 80)

    print(f"\nMessage count: " f"{len(raw_messages)}")

    for index, message in enumerate(raw_messages):
        print_message(index, message)

    # ---------------------------------------------------------
    # FINAL BUILDER OUTPUT
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print("FINAL ContextBuilder OUTPUT")
    print("=" * 80)

    print(f"\nMessage count: " f"{len(messages)}")

    for index, message in enumerate(messages):
        print_message(index, message)

    # ---------------------------------------------------------
    # TOKEN ANALYSIS
    # ---------------------------------------------------------

    raw_tokens = builder.tokenbudget.estimate_messages_tokens(raw_messages)

    final_tokens = builder.tokenbudget.estimate_messages_tokens(messages)

    print("\n" + "=" * 80)
    print("TOKEN ANALYSIS")
    print("=" * 80)

    print(f"\nRaw get_prompt tokens: " f"{raw_tokens}")

    print(f"Final context tokens: " f"{final_tokens}")

    print(f"Budget: " f"{builder.tokenbudget.budget}")

    print(f"Raw utilization: " f"{raw_tokens / builder.tokenbudget.budget:.2%}")

    print(f"Final utilization: " f"{final_tokens / builder.tokenbudget.budget:.2%}")

    # ---------------------------------------------------------
    # DUPLICATION CHECK
    # ---------------------------------------------------------

    print("\n" + "=" * 80)
    print("DUPLICATION CHECK")
    print("=" * 80)

    all_contents = [message.get("content", "") for message in messages]

    tool_output = "from flask import Flask"

    tool_output_count = sum(tool_output in content for content in all_contents)

    print(f"\nTool output occurrences: " f"{tool_output_count}")

    print("Expected: 1 " "(inside last_observation)")

    # ---------------------------------------------------------
    # ASSERTIONS
    # ---------------------------------------------------------

    assert messages
    assert isinstance(messages, list)

    for message in messages:
        assert "role" in message
        assert "content" in message

    assert builder.window.task == task
    assert builder.window.active_skills == active_skills
    assert builder.window.agent_state == agent_state
    assert builder.window.progress == progress

    assert builder.window.last_action["event_type"] == "tool_call"

    assert builder.window.last_observation["event_type"] == "tool_result"

    assert tool_output in builder.window.last_observation["content"]

    assert final_tokens <= builder.tokenbudget.budget

    print("\n" + "=" * 80)
    print("✅ ALL CONTEXT BUILDER TESTS PASSED")
    print("=" * 80)


if __name__ == "__main__":
    main()

[17:05:15] [INFO] [[PROVIDERREGISTRY]] → Found provider package: ollama
[17:05:15] [INFO] [[PROVIDERREGISTRY]] → Loaded module: <module 'src.engine.providers.builtin.ollama' from '/home/itsnxfi3/Desktop/Evana-agent-runtime/src/engine/providers/builtin/ollama/__init__.py'>
[17:05:15] [INFO] [[PROVIDERREGISTRY]] → Exports: ['OllamaProvider']
[17:05:15] [INFO] [[PROVIDERREGISTRY]] → Found provider class: <class 'src.engine.providers.builtin.ollama.ollama.OllamaProvider'>
[17:05:15] [INFO] [[PROVIDERREGISTRY]] → Instantiated: ollama
[17:05:15] [INFO] [[PROVIDERREGISTRY]] → Discovered 1 provider(s)
[17:05:15] [INFO] [[LLM]] → Loading ollama provider
[17:05:15] [INFO] [[TOKENBUDGET]] → Context length=120000, budget=103616, safe_margin=16384, initial_chars_per_token=4.0


EVANA CONTEXT BUILDER REAL TEST

[CONFIG]
Provider: ollama
Model: gpt-oss:20b
Context length: 120000

[LLM PROVIDER]
Loaded provider: ollama
Model object: OllamaProvider

[CONTEXT BUILDER]
Token budget: 103616
Safe margin: 16384
Initial chars/token: 4.0

BUILDING CONTEXT...

CONTEXT WINDOW STATE

SYSTEM:
{'os': 'Linux'}

TASK:
{'goal': 'Complete the Flask shop website'}

ACTIVE SKILLS:
{'filesystem': {'enabled': True}, 'python': {'enabled': True}}

AGENT STATE:
{'iteration': 5, 'status': 'working'}

PROGRESS:
{'completed': ['Inspected project structure', 'Inspected shop/app.py'],
 'failed': [],
 'pending': ['Modify shop/app.py', 'Add product images', 'Run tests']}

LAST ACTION:
{'content': '{"tool_name":"Read","arguments":{"path":"shop/app.py"}}',
 'event_type': 'tool_call',
 'metadata': {},
 'source': 'assistant',
 'step': 3,
 'timestamp': '2026-09-16 17:05:15.560527'}

LAST OBSERVATION:
{'content': 'from flask import Flask\n'
            '\n'
            'app = Flask(__name__)\n'
   

In [ ]:
context